# 개별종목 조합E — XGBoost

`기본모델/03.XGBoost.ipynb`과 같은 `models.xgboost.build_xgboost_baseline`을 가져오고
조합E 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.xgboost import build_xgboost_baseline  # noqa: E402

MODEL_NAME = 'XGBoost'
MODEL_BUILDER = build_xgboost_baseline


In [2]:
# 2. 조합E의 피처 값만 지정합니다.
import json

COMBINATION = 'E'
FEATURE_COLUMNS = (
    'sector_ret_5',
    'sector_ret_20',
    'relative_ret_5_sector',
    'relative_ret_20_sector',
    'relative_ret_5_market',
    'sector_hv_20',
    'sector_beta_60',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합E 피처: ('sector_ret_5', 'sector_ret_20', 'relative_ret_5_sector', 'relative_ret_20_sector', 'relative_ret_5_market', 'sector_hv_20', 'sector_beta_60')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4932,0.5012,-0.0080,0.3477,0.3740,0.0873,0.3694,0.1508,0.2601
1,2,balanced,980,20150123,20150421,0.3569,0.3978,-0.0409,0.2958,0.3236,-0.0110,0.3382,0.1527,0.2356
2,3,balanced,1210,20151228,20160328,0.3710,0.3762,-0.0051,0.3646,0.3649,0.0497,0.3672,0.2848,0.3352
3,4,balanced,1439,20161202,20170228,0.4371,0.4617,-0.0246,0.3526,0.3676,0.0611,0.3756,0.1962,0.2935
4,5,balanced,1669,20171113,20180207,0.4011,0.3901,0.0110,0.3820,0.3856,0.0811,0.3951,0.3603,0.3804
5,6,balanced,1899,20181024,20190118,0.3909,0.3725,0.0184,0.3888,0.3891,0.0868,0.4053,0.3481,0.3748
6,7,balanced,2129,20190930,20191224,0.4454,0.4781,-0.0328,0.3564,0.3719,0.0740,0.4000,0.2471,0.3298
7,8,balanced,2359,20200902,20201130,0.4034,0.3476,0.0558,0.3992,0.4003,0.0999,0.4034,0.3812,0.3944
8,9,balanced,2589,20210806,20211105,0.3691,0.3914,-0.0223,0.3621,0.3750,0.0570,0.3566,0.2726,0.3283
9,10,balanced,2818,20220714,20221012,0.3482,0.3454,0.0028,0.3467,0.3578,0.0354,0.3556,0.2438,0.3043


,OOS 폴드 평균
accuracy,0.4005
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0037
macro_f1,0.3642
balanced_accuracy,0.3743
mcc,0.0666
pr_auc_macro_ovr,0.3797
down_recall,0.2702
core_harmonic_mean,0.3291


재실행 명령: python scripts/run_stock_model_experiment.py
